<a href="https://colab.research.google.com/github/Emilycchi/googlecolabproject/blob/main/getprodrafts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

from google.colab import drive
drive.mount('/content/gdrive')

!pip install leaguepedia_parser
!pip install leaguepedia
!pip install wget
!pip install mwrogue
import wget
import mwrogue.esports_client
from mwrogue.esports_client import EsportsClient
import datetime as dt
import pandas as pd
from gspread_dataframe import set_with_dataframe
import leaguepedia_parser
from leaguepedia_parser.site.leaguepedia import leaguepedia
from leaguepedia_parser.transmuters.picks_bans import (
    picks_bans_fields,
    transmute_picks_bans,
)

# Uses the users google account to open and write sheets
gc = gspread.authorize(creds)

# Opens the relevant google sheets document
sh = gc.open_by_key('')
wks = sh.worksheet("RawData")

# Date (YYYY-MM-DD). Date decides the earliest Games to pull
date = "2026-04-15"
date = dt.datetime.strptime(date, "%Y-%m-%d").date()

# First query to get data from Leaguepedia
site = EsportsClient("lol")
response = site.cargo_client.query(
    tables="ScoreboardGames=SG, Tournaments=T",
    join_on="SG.OverviewPage=T.OverviewPage",
    fields="T.Name=Tournament, SG.DateTime_UTC, SG.Team1, SG.Team2, SG.WinTeam, SG.Patch, SG.VOD, SG.Winner, SG.GameId",
    where="SG.DateTime_UTC >= '" + str(date) + " 00:00:00'",
    having="T.Name LIKE 'LEC%' or T.Name LIKE 'LCK%' or T.Name LIKE 'LPL%' or T.Name LIKE 'LPLOL%' or T.Name LIKE 'MSI%' or T.Name LIKE 'Worlds%' or T.Name LIKE 'PRM 1st%' or T.Name LIKE 'ELITE SERIES%' <> T.Name LIKE 'ELITE SERIES 2' or T.Name LIKE 'LFL%' <> T.Name LIKE 'LFL Division 2%' or T.Name LIKE 'LVP SL%' or T.Name LIKE 'EM%' or T.Name LIKE 'NLC%'",
    order_by="SG.DateTime_UTC DESC"
)

# Transforms data into dataframe
Data = pd.DataFrame.from_dict(response)

# Gets Game-ID list and sum of games
GameIds = list(Data["GameId"])
size = len(GameIds)

# Creats a big dictionary with the drafts from every game
PickBan = []
for x in range(size):
    picks_bans = leaguepedia.query(
        tables="PicksAndBansS7, ScoreboardGames",
        join_on="PicksAndBansS7.GameId = ScoreboardGames.GameId",
        fields=", ".join(picks_bans_fields),
        where=f"ScoreboardGames.GameId = '{GameIds[x]}'",
    )
    if len(picks_bans):
      PickBan.extend(picks_bans)
    else:
      PickBan.extend([dict([('Team1Ban1', '-'), ('Team2Ban1', '-'), ('Team1Ban2', '-'), ('Team2Ban2', '-'), ('Team1Ban3', '-'), ('Team2Ban3', '-'), ('Team1Pick1', '-'), ('Team2Pick1', '-'), ('Team2Pick2', '-'), ('Team1Pick2', '-'), ('Team1Pick3', '-'), ('Team2Pick3', '-'), ('Team2Ban4', '-'), ('Team1Ban4', '-'), ('Team2Ban5', '-'), ('Team1Ban5', '-'), ('Team2Pick4', '-'), ('Team1Pick4', '-'), ('Team1Pick5', '-'), ('Team2Pick5', '-')])])

# Bundles all data into a final big dataframe
PickBans = pd.DataFrame.from_dict(PickBan)
FinalDataframe = pd.concat([Data, PickBans], axis=1)

# Writes the dataframe into the sheet
set_with_dataframe(wks, FinalDataframe.iloc[:,[0,1,2,3,4,5,6,7,8,9,10,12,14,23,25,11,13,15,22,24,16,19,20,27,28,17,18,21,26,29]])


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
ERROR: Could not find a version that satisfies the requirement leaguepedia (from versions: none)
ERROR: No matching distribution found for leaguepedia
